# 02. Ранкер LightGBM (v3)

**LB 0,8325** (тег `v3`). Продолжение `01_eda_baseline` (v2, LB 0,8313).

Что реализовано:
1. Валидация, согласованная с лидербордом: из двух схем выбираем ту, где результат v2 ближе всего к её LB.
2. Новые признаки и генератор кандидатов: символьные n-граммы (опечатки), поле «услуга» без адреса, город в тексте запроса, ранги кандидата в каждом списке пула.
3. Ранкер LightGBM вместо линейной формулы: обучаем на отдельных выборках запросов из train.
4. Ответ: предсказание для бенчмарка и `answer.csv`.

Время выполнения: ~45 мин на CPU, ~16 ГБ RAM.

## 0. Окружение

Подключаем код из `src/` и ставим недостающие пакеты (pymorphy3, LightGBM 4.6.0). Число потоков BLAS фиксируется до импорта numpy: так результат не зависит от машины.

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path

N_THREADS = 4
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = str(N_THREADS)

IS_KAGGLE = Path("/kaggle/input").exists()
REPO_URL = "https://github.com/mishin-mikhail/avito_autumn_dev.git"
REPO_REF = "main"          
REPO_DIR = Path("/kaggle/working/avito-candgen")


def github_token():
    """GITHUB_TOKEN из окружения или из Kaggle Secrets (None, если его нет)."""
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


def git(*args, token=None) -> str:
    """git без утечки токена: заголовок авторизации передаётся через переменные окружения."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"git завершился с ошибкой:\n{error}")
    return result.stdout.strip()


def checkout_repo() -> Path:
    """Клонирует (или обновляет) репозиторий вместе с тегами и переключается на REPO_REF."""
    token = github_token()
    if not REPO_DIR.exists():
        git("clone", "--quiet", REPO_URL, str(REPO_DIR), token=token)
    git("-C", str(REPO_DIR), "fetch", "--quiet", "--tags", "--force", "origin", token=token)
    is_branch = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--verify", "--quiet",
                                f"origin/{REPO_REF}"], capture_output=True).returncode == 0
    git("-C", str(REPO_DIR), "checkout", "--quiet", "--force", "--detach",
        f"origin/{REPO_REF}" if is_branch else REPO_REF)
    print("commit:", git("-C", str(REPO_DIR), "rev-parse", "HEAD"))
    return REPO_DIR


def find_repo_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:          
        if (path / "src" / "pipeline.py").exists():
            return path
    if not IS_KAGGLE:
        raise RuntimeError("Не найден код решения (src/). Запустите ноутбук внутри репозитория.")
    root = checkout_repo()
    if not (root / "src" / "pipeline.py").exists():
        raise RuntimeError(f"В корне репозитория нет src/pipeline.py: {sorted(p.name for p in root.iterdir())}")
    return root


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))


for module, packages in (("pymorphy3", ["pymorphy3==2.0.6", "pymorphy3-dicts-ru==2.4.417150.4580142"]),
                         ("lightgbm", ["lightgbm==4.6.0"])):
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("repo:", REPO_ROOT, "| kaggle:", IS_KAGGLE)

commit: 13f6690d9d55d46fa2082cf9abf2b55c6543c87e
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 79.3 MB/s eta 0:00:00
repo: /kaggle/working/avito-candgen | kaggle: True


In [2]:
import json
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import display

from src import analysis, eda
from src.candidates import BASE_FEATURES, build_vocabs
from src.config import CFG, RANKER_CFG
from src.data import load_benchmark, load_train, load_train_items_text
from src.paths import get_data_dir, get_output_dir, get_work_dir
from src.pipeline import STATS_COLS, ItemLemmaCache, add_lemma_keys, build_index, build_pool
from src.ranker import (add_stage1, importance_table, ranker_score, sample_training_rows, train_ranker)
from src.ranking import PoolData, linear_score, predict_from_scores, weights_vector
from src.repro import library_versions, seed_everything
from src.sampling import bench_targets, group_table, sample_queries
from src.submit import save_answer, validate_answer
from src.text import Lemmatizer
from src.utils import timer
from src.validation import add_query_segments, mark_seen, per_query_recall, recall_at_k

assert CFG.n_threads == N_THREADS
seed_everything(CFG.seed)
pd.set_option("display.max_colwidth", 100)

SMOKE = os.environ.get("SMOKE_TEST") == "1"
R = replace(RANKER_CFG, n_val_queries=300, fold_queries=300, max_rounds=150, early_stopping=30) if SMOKE else RANKER_CFG
DATA_DIR, WORK_DIR, OUT_DIR = get_data_dir(), get_work_dir(), get_output_dir()
VERSIONS = library_versions()
K, DEC = CFG.top_k, CFG.score_decimals
V2_W = weights_vector(BASE_FEATURES, R.v2_weights)
print(f"версия решения: {R.version}{' (SMOKE_TEST)' if SMOKE else ''}\ndata: {DATA_DIR}\nwork: {WORK_DIR}\nout:  {OUT_DIR}")
print(VERSIONS)

версия решения: v3
data: /kaggle/input/datasets/mm1khail/nlp-avito-interns/NLP_avito_interns/dataset
work: /kaggle/working/artifacts
out:  /kaggle/working
{'python': '3.12.13', 'numpy': '2.0.2', 'pandas': '2.3.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'pyarrow': '24.0.0', 'pymorphy3': '2.0.6', 'lightgbm': '4.6.0', 'torch': '2.10.0+cpu'}


## 1. Данные

Как в v2. Дополнительно для каждого запроса train отмечаем, лежат ли все выбранные по нему объявления в корпусе бенчмарка: это нужно для второй схемы валидации.

In [3]:
with timer("загрузка"):
    train = load_train(DATA_DIR)
    bench_q, bench_items = load_benchmark(DATA_DIR)

lem = Lemmatizer()
with timer("подготовка запросов"):
    add_lemma_keys(train, lem)
    add_lemma_keys(bench_q, lem)
    item_locations = pd.Index(pd.concat([train["item_location_id"], bench_items["item_location_id"]]).unique())
    add_query_segments(train, item_locations)
    add_query_segments(bench_q, item_locations)
    mark_seen(bench_q, train["norm_text"].unique())
    groups = group_table(train, bench_items["item_id"])

OVERLAP = eda.overlap(train, bench_q, bench_items, CFG.item_stats_min_overlap)
print(f"\nгрупп-запросов: {len(groups):,}; из них все выбранные объявления в корпусе: "
      f"{groups['in_corpus'].sum():,} ({groups['in_corpus'].mean():.3f})")

[загрузка] 32.7 c
[подготовка запросов] 6.7 c

── Пересечение бенчмарка с train ─────────────────────────────────────────
объявлений корпуса, встречающихся в train: 0.096
запросов, чей текст встречается в train:  0.375
запросов, чей полный ключ есть в train:   0.044
→ статистики по item_id выключены (порог 0.5)

групп-запросов: 354,241; из них все выбранные объявления в корпусе: 18,415 (0.052)


## 2. Выборки валидации

Валидация v2 оказалась легче бенчмарка: 0,877 против 0,831 на LB. Гипотеза: корпус бенчмарка собран вокруг его запросов, поэтому у настоящих эталонов много похожих конкурентов, а у эталонов, подмешанных в корпус, мало. Сравниваем две схемы с одинаковым составом:
* `injected`: любые запросы train, их эталоны подмешиваются в корпус (как в v2);
* `in_corpus`: только запросы, чьи эталоны уже лежат в корпусе бенчмарка, со своим настоящим окружением.

Новые тексты теперь берём по одному запросу на текст: в бенчмарке новые тексты почти всегда редкие.

In [4]:
ALL_ROWS = np.ones(len(train), dtype=bool)
KEYS = {"injected": pd.Index(groups["query_key"]),
        "in_corpus": pd.Index(groups.loc[groups["in_corpus"], "query_key"])}
val_target = bench_targets(bench_q, R.n_val_queries)

with timer("выборки валидации"):
    VAL = {name: sample_queries(f"валидация {name}", train, groups, val_target, ALL_ROWS, keys,
                                R.text_holdout_frac, R.val_salt)
           for name, keys in KEYS.items()}

pd.concat({name: s.report.set_index(["текст", "страта"])["факт"] for name, s in VAL.items()},
          axis=1).assign(цель=val_target.to_numpy())

[warn] валидация in_corpus: в некоторых ячейках не хватило запросов — 2274 из 2500
[выборки валидации] 10.4 c


injected  in_corpus  цель
текст    страта                                                           
знакомый фильтр есть | локация обычная                443        443   443
         фильтр есть | локация только поисковая        84         84    84
         фильтра нет | локация обычная                333        333   333
         фильтра нет | локация только поисковая        78         78    78
новый    фильтр есть | локация обычная                336        336   336
         фильтр есть | локация только поисковая        61         61    61
         фильтра нет | локация обычная                954        782   954
         фильтра нет | локация только поисковая       211        157   211

Для `in_corpus` не хватает новых запросов без фильтра: 2 274 запроса вместо 2 500.

## 3. Выбор схемы валидации

Строим индексы корпуса бенчмарка и его копии с подмешанными эталонами, затем пулы кандидатов для обеих схем и для бенчмарка.

In [ ]:
vocabs = build_vocabs([train, bench_q], [train, bench_items])
cache = ItemLemmaCache(lem, CFG.desc_max_chars)
bench_ids = frozenset(bench_items["item_id"])       


def items_with_injected(samples) -> pd.DataFrame:
    """Корпус бенчмарка + эталонные объявления выборок, которых в нём нет."""
    missing = sorted(dict.fromkeys(i for s in samples for rel in s.truth for i in rel if i not in bench_ids))
    extra = load_train_items_text(DATA_DIR, missing)
    print(f"подмешано объявлений: {len(extra):,}")
    return pd.concat([bench_items, extra[bench_items.columns]], ignore_index=True)


def make_pool(name, queries, truth, index, items, stats_mask):
    """Пул v3 с признаками и скором формулы v2 (stage1)."""
    _, pool = build_pool(name, index, items, queries, train.loc[stats_mask, STATS_COLS], lem=lem,
                         vocabs=vocabs, cfg=CFG, use_item_stats=OVERLAP["use_item_stats"],
                         truth=truth, ext_cfg=R)
    return add_stage1(pool, index, R.v2_weights, DEC)


bench_index = build_index("корпус бенчмарка", bench_items, cache=cache, vocabs=vocabs, cfg=CFG, ext_cfg=R)
inj_items = items_with_injected([VAL["injected"]])
inj_index = build_index("корпус injected", inj_items, cache=cache, vocabs=vocabs, cfg=CFG, ext_cfg=R)
CORPORA = {"injected": (inj_index, inj_items), "in_corpus": (bench_index, bench_items)}

VAL_POOLS = {name: make_pool(f"валидация {name}", s.queries, s.truth, *CORPORA[name], s.stats_mask)
             for name, s in VAL.items()}
BENCH_POOL = make_pool("бенчмарк", bench_q, None, bench_index, bench_items, ALL_ROWS)

[корпус бенчмарка: индекс корпуса] 281.8 c
подмешано объявлений: 2,779
[корпус injected: индекс корпуса] 103.3 c
  пул: 64/2500 запросов
  пул: 384/2500 запросов
  пул: 704/2500 запросов
  пул: 1024/2500 запросов
  пул: 1344/2500 запросов
  пул: 1664/2500 запросов
  пул: 1984/2500 запросов
  пул: 2304/2500 запросов
[валидация injected: статистики и пул] 103.0 c
валидация injected: запросов 2,500, строк пула 2,546,856 (~1019 на запрос)
  пул: 64/2274 запросов
  пул: 384/2274 запросов
  пул: 704/2274 запросов
  пул: 1024/2274 запросов
  пул: 1344/2274 запросов
  пул: 1664/2274 запросов
  пул: 1984/2274 запросов
  пул: 2274/2274 запросов
[валидация in_corpus: статистики и пул] 96.5 c
валидация in_corpus: запросов 2,274, строк пула 2,405,501 (~1058 на запрос)
  пул: 64/2452 запросов
  пул: 384/2452 запросов
  пул: 704/2452 запросов
  пул: 1024/2452 запросов
  пул: 1344/2452 запросов
  пул: 1664/2452 запросов
  пул: 1984/2452 запросов
  пул: 2304/2452 запросов
[бенчмарк: статистики и пул] 1

Для каждой схемы считаем Recall@50 ровно той модели, что была отправлена как v2, и сравниваем с её LB. Берём схему с наименьшим расхождением. Вторая таблица показывает плотность конкуренции: сколько у запроса объявлений в той же локации и рядом.

In [6]:
calibration = []
for name, s in VAL.items():
    pool, (index, _) = VAL_POOLS[name], CORPORA[name]
    v2_pool = analysis.v2_rows(pool)
    recall_v2 = PoolData(v2_pool, index, BASE_FEATURES, s.n_rel).recall(V2_W, K, DEC)
    calibration.append({
        "схема": name, "запросов": len(s.queries), "Recall@50 v2": recall_v2,
        "LB v2": R.v2_lb, "расхождение": recall_v2 - R.v2_lb,
        "полнота пула v2": analysis.pool_hit_rate(v2_pool, s.n_rel).mean(),
        "полнота пула v3": analysis.pool_hit_rate(pool, s.n_rel).mean(),
    })
calibration = pd.DataFrame(calibration).set_index("схема").round(4)
SCHEME = calibration["расхождение"].abs().idxmin() if R.val_scheme == "auto" else R.val_scheme
display(calibration)
print(f"→ схема валидации: {SCHEME}")

display(analysis.difficulty_table({"бенчмарк": BENCH_POOL, **{f"валидация {n}": p for n, p in VAL_POOLS.items()}}, K))

,запросов,Recall@50 v2,LB v2,расхождение,полнота пула v2,полнота пула v3
схема,,,,,,
injected,2500,0.8634,0.8313,0.0321,0.9579,0.9619
in_corpus,2274,0.8748,0.8313,0.0435,0.9693,0.9737


→ схема валидации: injected


,запросов,объявлений в той же локации,объявлений рядом,max BM25 заголовка,скор v2 у 50-го,доля «региональных»
бенчмарк,2452.0,1638.0,3259.0,14.844,4.585,0.174
валидация injected,2500.0,697.0,1917.0,14.327,3.799,0.175
валидация in_corpus,2274.0,1157.0,2709.0,14.667,4.221,0.167


Обе схемы завышают результат v2: на 0,032 (`injected`) и 0,044 (`in_corpus`), выбрана `injected`. В v2 расхождение было 0,046: выбор новых текстов по одному на текст приблизил валидацию к бенчмарку.

Гипотеза подтверждается: в бенчмарке у запроса 1 638 объявлений в той же локации, на валидации 697. Полностью воспроизвести такую конкуренцию на валидации нельзя.

Ниже проверка: пайплайн v3 без новых кандидатов и с весами v2 побайтно воспроизводит отправленный ответ v2.

In [ ]:
v2_bench = analysis.v2_rows(BENCH_POOL)
v2_pred = predict_from_scores(v2_bench, bench_index, linear_score(v2_bench[BASE_FEATURES].to_numpy(np.float64), V2_W),
                              K, DEC, len(bench_q))
v2_md5 = save_answer(bench_q["query_id"], v2_pred, WORK_DIR / "answer_v2_check.csv")
print(f"md5 ответа v2 из пайплайна v3: {v2_md5} | отправленный: {R.v2_answer_md5} | "
      f"совпадает: {v2_md5 == R.v2_answer_md5}")

md5 ответа v2 из пайплайна v3: 2de61da58afd87fc244e225b7cccde58 | отправленный: 2de61da58afd87fc244e225b7cccde58 | совпадает: True


## 4. Фолды для ранкера

Запросы для обучения отбираем так же, как валидацию, но без пересечения с ней. Статистики каждого фолда (например, P(микрокатегория | запрос)) считаются без его строк, иначе признаки подсказывали бы ответ.

Три фолда идут в обучение: все позитивы плюс трудные, символьно похожие и случайные негативы. Четвёртый фолд целиком используется для ранней остановки.

In [ ]:
eligible = KEYS[SCHEME].difference(pd.Index(VAL[SCHEME].keys))
fold_target = bench_targets(bench_q, R.fold_queries)
FOLDS = []
with timer("выборки фолдов"):
    for f in range(R.n_folds):
        fold = sample_queries(f"фолд {f}", train, groups, fold_target, VAL[SCHEME].stats_mask, eligible,
                              R.fold_holdout_frac, f"{R.val_salt}-fold{f}")
        eligible = eligible.difference(pd.Index(fold.keys))
        FOLDS.append(fold)
print("запросов в фолдах:", [len(f.queries) for f in FOLDS])

if SCHEME == "injected":        
    fold_items = items_with_injected([VAL["injected"], *FOLDS])
    fold_index = build_index("корпус injected + фолды", fold_items, cache=cache, vocabs=vocabs, cfg=CFG, ext_cfg=R)
    CORPORA["injected"] = (fold_index, fold_items)
    VAL_POOLS["injected"] = make_pool("валидация injected", VAL["injected"].queries, VAL["injected"].truth,
                                      fold_index, fold_items, VAL["injected"].stats_mask)
else:
    fold_index, fold_items = bench_index, bench_items

[выборки фолдов] 20.3 c
запросов в фолдах: [4000, 4000, 4000, 4000]
подмешано объявлений: 20,136
[корпус injected + фолды: индекс корпуса] 129.3 c
  пул: 64/2500 запросов
  пул: 384/2500 запросов
  пул: 704/2500 запросов
  пул: 1024/2500 запросов
  пул: 1344/2500 запросов
  пул: 1664/2500 запросов
  пул: 1984/2500 запросов
  пул: 2304/2500 запросов
[валидация injected: статистики и пул] 126.3 c
валидация injected: запросов 2,500, строк пула 2,568,025 (~1027 на запрос)


In [9]:
train_parts = []
for f, fold in enumerate(FOLDS):
    pool = make_pool(f"фолд {f}", fold.queries, fold.truth, fold_index, fold_items, fold.stats_mask)
    if f < R.n_folds - 1:
        rows = sample_training_rows(pool, R, seed=CFG.seed + f)
        rows.insert(0, "gid", f * 1_000_000 + rows["q"].astype(np.int64))
        train_parts.append(rows)
    else:
        VALID_POOL = pool
    del pool

TRAIN_ROWS = pd.concat(train_parts, ignore_index=True)
del train_parts
print(f"обучающих строк: {len(TRAIN_ROWS):,} (позитивов {int(TRAIN_ROWS['label'].sum()):,}); "
      f"строк в фолде остановки: {len(VALID_POOL):,}")

  пул: 64/4000 запросов
  пул: 384/4000 запросов
  пул: 704/4000 запросов
  пул: 1024/4000 запросов
  пул: 1344/4000 запросов
  пул: 1664/4000 запросов
  пул: 1984/4000 запросов
  пул: 2304/4000 запросов
  пул: 2624/4000 запросов
  пул: 2944/4000 запросов
  пул: 3264/4000 запросов
  пул: 3584/4000 запросов
  пул: 3904/4000 запросов
[фолд 0: статистики и пул] 192.6 c
фолд 0: запросов 4,000, строк пула 4,127,957 (~1032 на запрос)
  пул: 64/4000 запросов
  пул: 384/4000 запросов
  пул: 704/4000 запросов
  пул: 1024/4000 запросов
  пул: 1344/4000 запросов
  пул: 1664/4000 запросов
  пул: 1984/4000 запросов
  пул: 2304/4000 запросов
  пул: 2624/4000 запросов
  пул: 2944/4000 запросов
  пул: 3264/4000 запросов
  пул: 3584/4000 запросов
  пул: 3904/4000 запросов
[фолд 1: статистики и пул] 181.9 c
фолд 1: запросов 4,000, строк пула 4,110,131 (~1028 на запрос)
  пул: 64/4000 запросов
  пул: 384/4000 запросов
  пул: 704/4000 запросов
  пул: 1024/4000 запросов
  пул: 1344/4000 запросов
  пул: 166

## 5. Обучение ранкера

Обучаем LightGBM с двумя целевыми функциями, `lambdarank` и `binary`. Ранняя остановка и выбор между ними: по Recall@50 на четвёртом фолде. Валидация в обучении не участвует.

In [10]:
valid_fold = FOLDS[-1]
valid_rank = fold_index.rank[VALID_POOL["item"].to_numpy()]
fold_scores = {"формула v2": PoolData(VALID_POOL, fold_index, BASE_FEATURES, valid_fold.n_rel).recall(V2_W, K, DEC)}
MODELS = {}
for objective in R.objectives:
    with timer(f"LightGBM {objective}"):
        booster, best_iter, best = train_ranker(TRAIN_ROWS, VALID_POOL, valid_fold.n_rel, valid_rank,
                                                objective, R, CFG.seed, N_THREADS, K, DEC)
    MODELS[objective] = booster
    fold_scores[f"ранкер {objective}"] = best
    print(f"{objective}: лучшая итерация {best_iter}, Recall@{K} на фолде {best:.4f}")

OBJECTIVE = max(R.objectives, key=lambda o: fold_scores[f"ранкер {o}"])
RANKER = MODELS[OBJECTIVE]
pd.Series(fold_scores, name=f"Recall@{K} на фолде остановки").round(4).to_frame()

[100]	fold's recall@50: 0.90968
[200]	fold's recall@50: 0.908924
[300]	fold's recall@50: 0.910138
[400]	fold's recall@50: 0.910519
[LightGBM lambdarank] 581.3 c
lambdarank: лучшая итерация 309, Recall@50 на фолде 0.9110
[100]	fold's recall@50: 0.907619
[200]	fold's recall@50: 0.907821
[LightGBM binary] 240.9 c
binary: лучшая итерация 128, Recall@50 на фолде 0.9093


,Recall@50 на фолде остановки
формула v2,0.8678
ранкер lambdarank,0.9110
ранкер binary,0.9093


Ранкер поднимает Recall@50 на фолде остановки с 0,868 (формула v2) до 0,911. Лучший: `lambdarank`.

## 6. Качество на валидации

Валидация не участвовала ни в обучении, ни в ранней остановке, поэтому это честная оценка.

In [11]:
val_s, val_pool = VAL[SCHEME], VAL_POOLS[SCHEME]
val_index = CORPORA[SCHEME][0]
nq = len(val_s.queries)


def val_recall(pool, score):
    return recall_at_k(predict_from_scores(pool, val_index, score, K, DEC, nq), val_s.truth, K)


v2_pool = analysis.v2_rows(val_pool)
ranker_val_score = ranker_score(RANKER, val_pool, N_THREADS)
VAL_RESULTS = {
    "v2: пул v2 + формула v2": val_recall(v2_pool, v2_pool["stage1"].to_numpy(np.float64)),
    "формула v2 на пуле v3": val_recall(val_pool, val_pool["stage1"].to_numpy(np.float64)),
    f"ранкер ({OBJECTIVE})": val_recall(val_pool, ranker_val_score),
}
USE_RANKER = VAL_RESULTS[f"ранкер ({OBJECTIVE})"] > VAL_RESULTS["формула v2 на пуле v3"]
display(pd.Series(VAL_RESULTS, name=f"Recall@{K}, валидация {SCHEME}").round(4).to_frame())
print(f"полнота пула v3: {analysis.pool_hit_rate(val_pool, val_s.n_rel).mean():.4f} | "
      f"в ответ идёт: {'ранкер' if USE_RANKER else 'формула v2'}")

,"Recall@50, валидация injected"
v2: пул v2 + формула v2,0.8587
формула v2 на пуле v3,0.8599
ранкер (lambdarank),0.8970


полнота пула v3: 0.9612 | в ответ идёт: ранкер


In [12]:
final_val_score = ranker_val_score if USE_RANKER else val_pool["stage1"].to_numpy(np.float64)
val_pred = predict_from_scores(val_pool, val_index, final_val_score, K, DEC, nq)
r50 = per_query_recall(val_pred, val_s.truth, K)
analysis.segment_table(val_s.queries, r50, analysis.pool_hit_rate(val_pool, val_s.n_rel))

n   share    pool  recall50
ось        сегмент                                                 
seg_text   знакомый                   938  0.3752  0.9757    0.9316
           новый                     1562  0.6248  0.9524    0.8762
seg_filter фильтр есть                924  0.3696  0.9685    0.9250
           фильтра нет               1576  0.6304  0.9569    0.8806
seg_loc    локация обычная           2066  0.8264  0.9791    0.9244
           локация только поисковая   434  0.1736  0.8760    0.7668

In [13]:
display(importance_table(RANKER).head(25))
errors = analysis.error_examples(val_pred, val_s.truth, val_s.queries, CORPORA[SCHEME][1],
                                 analysis.pool_hit_rate(val_pool, val_s.n_rel), n=15)
print(f"запросов без единого попадания: {int((r50 == 0).sum())} ({(r50 == 0).mean():.3f})")
errors

,признак,gain,доля
0,stage1_rank,1.072342e+06,0.595497
1,rank_text_loc,2.048186e+05,0.113741
2,desc,7.676134e+04,0.042627
3,log_dist,7.425112e+04,0.041233
4,stage1,3.355628e+04,0.018635
5,char,3.218709e+04,0.017874
6,rank_text,3.137784e+04,0.017425
7,rank_char_loc,2.329187e+04,0.012935
8,rank_prior_loc,2.323613e+04,0.012904
9,q_n_near,2.241929e+04,0.012450


запросов без единого попадания: 242 (0.097)


,запрос,фильтры,та же локация,в пуле,заголовок эталона,параметры эталона
0,утилизация газовых плит бесплатно,Вид услуги Вывоз мусора и вторсырья,False,False,Вывоз старой мебели и бытовой техники грузчики,"Вид услуги Вывоз мусора и вторсырья Место оказания услуг Москва, 2-й Митинский переулок, 5, подъ..."
1,массаж,"Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",False,True,Классический расслабляющий массаж всего тела,"Вид услуги Красота, здоровье Место оказания услуг Республика Татарстан, Казань, улица Юлиуса Фуч..."
2,похудение,"Онлайн-запись Вид услуги Красота, здоровье",True,True,LPG массаж. Антицеллюлитный массаж,"Вид услуги Красота, здоровье Место оказания услуг Свердловская область, Екатеринбург, проспект К..."
3,заливка бетона,,False,True,Фундамент Отмостка Парковка,"Вид услуги Строительство Место оказания услуг Нижний Новгород, Приокский район, Цветочная улица,..."
4,ремонт фанкойлов,,False,False,"Установка и Продажа Кондиционеров,Ремонт,Заправка","Вид услуги Монтаж и установка техники Место оказания услуг Санкт-Петербург, улица Дыбенко, 26 Ти..."
5,ремонт компьютеров,Вид услуги Компьютерная помощь,True,True,Ком Тех,"Вид услуги Деловые услуги Тип услуги Место оказания услуг Краснодарский край, станица Выселки Ти..."
6,аренда авто с водителем,Вид услуги Пассажирские перевозки,True,True,Аренда автомобиля с водителем,"Вид услуги Пассажирские перевозки Место оказания услуг Москва, улица Александра Солженицына Марк..."
7,прокат мото,,False,False,Прокат снегоходов / квадроциклов / эндуро,"Вид услуги Праздники, мероприятия Место оказания услуг Республика Башкортостан, Абзелиловский ра..."
8,прием металлолома чермед,Вид услуги Вывоз мусора и вторсырья,False,False,Утилизация демонтаж вывоз,"Вид услуги Вывоз мусора и вторсырья Место оказания услуг Удмуртская Республика, Сарапул, Централ..."
9,пескоструйная обработка рамы авто,,True,True,"Пескоструйная обработка, рам, арок","Вид услуги Оборудование, производство Тип услуги Производство, обработка Место оказания услуг ра..."


**Итог:** ранкер даёт 0,897 против 0,860 у формулы v2 на том же пуле (+0,037). Важнее всего для него ранг по формуле v2 (60%) и ранг в текстовом списке с приоритетом близости (11%).

На лидерборде прирост оказался всего +0,001 (0,8325). Улучшение ранжирования на бенчмарк не перенеслось, поэтому в следующих версиях главная цель: полнота пула.

## 7. Бенчмарк

Пул бенчмарка построен в разделе 3 по статистикам всего train. Файл проверяется на формат, как в v2.

In [14]:
bench_score = ranker_score(RANKER, BENCH_POOL, N_THREADS) if USE_RANKER else BENCH_POOL["stage1"].to_numpy(np.float64)
bench_pred = predict_from_scores(BENCH_POOL, bench_index, bench_score, K, DEC, len(bench_q))

ANSWER_PATH = OUT_DIR / "answer.csv"
save_answer(bench_q["query_id"], bench_pred, ANSWER_PATH)
CHECK = validate_answer(ANSWER_PATH, bench_q["query_id"], bench_items["item_id"], K)
print(CHECK)
overlap_v2 = np.mean([len(frozenset(a) & frozenset(b)) / K for a, b in zip(bench_pred, v2_pred)])
print(f"совпадение с ответом v2: {overlap_v2:.3f} объявлений из топ-50 в среднем")

{'rows': 2452, 'min_items': 50, 'max_items': 50, 'md5': '93003388ccf17692af66c8f8a38fc98c'}
совпадение с ответом v2: 0.594 объявлений из топ-50 в среднем


## 8. Артефакты

Модель в текстовом формате LightGBM, отчёт с метриками, конфигами и версиями библиотек. Повторный запуск даёт тот же md5 `answer.csv`.

In [15]:
RANKER.save_model(str(WORK_DIR / f"ranker_{R.version}.txt"), num_iteration=RANKER.best_iteration)
report = {
    "version": R.version,
    "answer_md5": CHECK["md5"],
    "val_scheme": SCHEME,
    "calibration": calibration.reset_index().to_dict(orient="records"),
    "v2_reproduced": v2_md5 == R.v2_answer_md5,
    "val_recall@50": VAL_RESULTS,
    "fold_recall@50": fold_scores,
    "objective": OBJECTIVE,
    "best_iteration": RANKER.best_iteration,
    "use_ranker": bool(USE_RANKER),
    "config": CFG.as_dict(),
    "ranker_config": R.as_dict(),
    "versions": VERSIONS,
}
(WORK_DIR / f"report_{R.version}.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str))

for name, value in VAL_RESULTS.items():
    print(f"{name:28s} {value:.4f}")
print(f"answer.csv md5: {CHECK['md5']}")

v2: пул v2 + формула v2      0.8587
формула v2 на пуле v3        0.8599
ранкер (lambdarank)          0.8970
answer.csv md5: 93003388ccf17692af66c8f8a38fc98c
